In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, RandomTranslation, Input, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt

In [ ]:

# --- STEP 1: CONFIGURATION  ---
IMAGE_SIZE = (300, 300)  
BATCH_SIZE = 8           
EPOCHS_PHASE_1 = 10      
EPOCHS_PHASE_2 = 10      

base_dir = '/content/Final_Dataset/'
TRAIN_DIR = os.path.join(base_dir, 'train')
VAL_DIR = os.path.join(base_dir, 'test')

print("✅ Setup Complete. Libraries Imported.")

In [ ]:
# --- STEP 2: DATA PIPELINE  ---

# 1. Training Data 
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode='categorical', 
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# 2. Validation Data (Ise hum clean rakhenge)
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    label_mode='categorical',
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print(f"🔍 Total Classes Found: {NUM_CLASSES}")
print(class_names)

# 3. Data Augmentation Layer 
data_augmentation = tf.keras.Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.2),
    RandomZoom(0.2),
    RandomTranslation(height_factor=0.1, width_factor=0.1),
], name="data_augmentation")

# 4. Preprocessing Functions
def preprocess_train(image, label):
    image = data_augmentation(image, training=True)
    return image, label

def preprocess_val(image, label):
    return image, label

# 5. Pipeline Optimization (Speed badhana)
train_generator = train_ds.map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
validation_generator = val_ds.map(preprocess_val, num_parallel_calls=tf.data.AUTOTUNE)

train_generator = train_generator.prefetch(buffer_size=tf.data.AUTOTUNE)
validation_generator = validation_generator.prefetch(buffer_size=tf.data.AUTOTUNE)

print("✅ Data Pipeline Ready with Augmentation & Prefetching.")


In [ ]:
# --- STEP 3: MODEL ARCHITECTURE ---

# 1. Base Model 
base_model = EfficientNetB3(
    input_shape=(300, 300, 3),
    include_top=False,    
    weights='imagenet' 
)

# 2. Custom Head 
inputs = Input(shape=(300, 300, 3))

x = base_model(inputs, training=False) 

x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x) 
x = Dropout(0.5)(x)                  
predictions = Dense(NUM_CLASSES, activation='softmax')(x) 

# 3. Final Model
model = Model(inputs=inputs, outputs=predictions)

print("✅ Model Built.")



In [ ]:
# --- STEP 4: PHASE 1 TRAINING  ---

print("\n🚀 Starting Phase 1: Training Head only...")

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Checkpoints 
checkpoint_path_1 = "/content/drive/My Drive/crop_model_phase1.weights.h5"
checkpoint_cb_1 = ModelCheckpoint(
    checkpoint_path_1, save_weights_only=True, monitor='val_accuracy', save_best_only=True
)
early_stop_cb = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

# Training Run
history_1 = model.fit(
    train_generator,
    epochs=EPOCHS_PHASE_1,
    validation_data=validation_generator,
    callbacks=[checkpoint_cb_1, early_stop_cb]
)



In [ ]:

# --- STEP 5: PHASE 2 TRAINING (Fine-Tuning) ---


fine_tune_at = 250
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# New Checkpoint for Final Model
final_weights_path = "/content/drive/My Drive/crop_model_finetuned.weights.h5"
checkpoint_cb_2 = ModelCheckpoint(
    final_weights_path, save_weights_only=True, monitor='val_accuracy', save_best_only=True, mode='max'
)

# Training Run (Resume from Phase 1)
history_2 = model.fit(
    train_generator,
    epochs=EPOCHS_PHASE_2, # Total extra epochs
    validation_data=validation_generator,
    callbacks=[checkpoint_cb_2, early_stop_cb]
)

print("✅ Training Complete!")


In [ ]:
# --- STEP 6: SAVING FINAL FULL MODEL (For App/API) ---

print("\n💾 Saving Final Full Model for App...")
full_model_save_path = "/content/drive/My Drive/FasalSarthi_Full_Model.h5"
model.save(full_model_save_path)

print(f"🎉 SUCCESS! Model saved at: {full_model_save_path}")